In [186]:
import pandas as pd
import numpy as np

# -------------------------------------------------------------------
# 1) Load + normalize inputs
# -------------------------------------------------------------------

# Snapshots (pid_snaps)
df_snaps = pd.read_csv("pid_snaps.csv")
df_snaps['snpsht_dt'] = pd.to_datetime(df_snaps['snpsht_dt']).dt.normalize()

# OERs (ofcr_oers)
df_oers = pd.read_csv("ofcr_oers.csv")

# Quick schema check
print("pid_snaps columns:", list(df_snaps.columns))
print("ofcr_oers columns:", list(df_oers.columns))

# Normalize date columns
if 'eval_strt_dt' in df_oers.columns:
    df_oers['eval_strt_dt'] = pd.to_datetime(df_oers['eval_strt_dt']).dt.normalize()
if 'eval_thru_dt' in df_oers.columns:
    df_oers['eval_thru_dt'] = pd.to_datetime(df_oers['eval_thru_dt']).dt.normalize()

# Standardize officer id column name
df_oers = df_oers.rename(columns={'rated_ofcr': 'pid_pde'})

# Allow either rtr_rater or rater naming
if 'rtr_rater' in df_oers.columns and 'rater' not in df_oers.columns:
    df_oers = df_oers.rename(columns={'rtr_rater': 'rater'})

# -------------------------------------------------------------------
# 2) Assign OERs to snapshots (forward + backward)
# -------------------------------------------------------------------

def assign_oer_to_snaps(snaps: pd.DataFrame, oers: pd.DataFrame) -> pd.DataFrame:
    """
    Forward: most recent eval_thru_dt <= snpsht_dt
    Backward: eval_strt_dt <= snpsht_dt <= eval_thru_dt
    """
    # merge_asof requires the ON key to be globally sorted
    snaps_sorted = snaps.sort_values(['snpsht_dt', 'pid_pde']).reset_index(drop=True)
    oers_thru_sorted = oers.sort_values(['eval_thru_dt', 'pid_pde']).reset_index(drop=True)

    # --- Forward assignment (carry-forward) ---
    df_fwd = pd.merge_asof(
        snaps_sorted,
        oers_thru_sorted,
        left_on='snpsht_dt',
        right_on='eval_thru_dt',
        by='pid_pde',
        direction='backward',
    )

    fwd_cols = {
        'rater': 'rtr_rater_fwd',
        'snr_rater': 'snr_rater_fwd',
        'rtr_rater_box': 'rtr_rater_box_fwd',
        'snr_rater_box': 'snr_rater_box_fwd',
        'oer_id': 'oer_id_fwd',
        'eval_strt_dt': 'eval_strt_dt_fwd',
        'eval_thru_dt': 'eval_thru_dt_fwd',
    }
    df_fwd = df_fwd.rename(columns=fwd_cols)

    # --- Backward assignment (painted) ---
    # Use eval_thru_dt as the align key, then filter outside-window rows
    df_bwd = pd.merge_asof(
        snaps_sorted,
        oers_thru_sorted,
        left_on='snpsht_dt',
        right_on='eval_thru_dt',
        by='pid_pde',
        direction='forward',
    )

    mask_outside = df_bwd['snpsht_dt'] < df_bwd['eval_strt_dt']
    for col in ['rater', 'snr_rater', 'rtr_rater_box', 'snr_rater_box', 'oer_id', 'eval_strt_dt', 'eval_thru_dt']:
        df_bwd.loc[mask_outside, col] = np.nan

    bwd_cols = {
        'rater': 'rtr_rater_bwd',
        'snr_rater': 'snr_rater_bwd',
        'rtr_rater_box': 'rtr_rater_box_bwd',
        'snr_rater_box': 'snr_rater_box_bwd',
        'oer_id': 'oer_id_bwd',
        'eval_strt_dt': 'eval_strt_dt_bwd',
        'eval_thru_dt': 'eval_thru_dt_bwd',
    }
    df_bwd = df_bwd.rename(columns=bwd_cols)

    # Keep forward + backward columns together
    merge_cols = ['pid_pde', 'snpsht_dt'] + list(bwd_cols.values())
    df_tb = df_fwd.merge(df_bwd[merge_cols], on=['pid_pde', 'snpsht_dt'], how='left')

    # Normalize merged date columns (merge_asof can yield object dtype)
    for col in ['eval_strt_dt_fwd', 'eval_thru_dt_fwd', 'eval_strt_dt_bwd', 'eval_thru_dt_bwd']:
        if col in df_tb.columns:
            df_tb[col] = pd.to_datetime(df_tb[col]).dt.normalize()

    # Paint backward columns across the full rated period
    bwd_fill_cols = [
        'rtr_rater_bwd', 'snr_rater_bwd', 'rtr_rater_box_bwd', 'snr_rater_box_bwd',
        'oer_id_bwd', 'eval_strt_dt_bwd', 'eval_thru_dt_bwd'
    ]
    df_tb[bwd_fill_cols] = (
        df_tb.groupby(['pid_pde', 'eval_thru_dt_bwd'])[bwd_fill_cols]
        .transform('first')
    )

    return df_tb


# Build combined snapshot/OER frame

df_tb = assign_oer_to_snaps(df_snaps, df_oers)

# -------------------------------------------------------------------
# 3) Cumulative TB metrics (forward + backward)
# -------------------------------------------------------------------

TB_VALUE = 70  # only 70 counts as a top block

# Forward cumulative counts at eval_thru_dt
is_eval_end_fwd = df_tb['snpsht_dt'] == df_tb['eval_thru_dt_fwd']

df_tb['cum_evals_fwd_rtr'] = is_eval_end_fwd.groupby(df_tb['pid_pde']).cumsum()
df_tb['cum_tb_fwd_rtr'] = ((df_tb['rtr_rater_box_fwd'] == TB_VALUE) & is_eval_end_fwd).groupby(df_tb['pid_pde']).cumsum()

df_tb['cum_evals_fwd_snr'] = is_eval_end_fwd.groupby(df_tb['pid_pde']).cumsum()
df_tb['cum_tb_fwd_snr'] = ((df_tb['snr_rater_box_fwd'] == TB_VALUE) & is_eval_end_fwd).groupby(df_tb['pid_pde']).cumsum()

df_tb['TB_ratio_fwd_rtr'] = np.where(
    df_tb['cum_evals_fwd_rtr'] > 0,
    df_tb['cum_tb_fwd_rtr'] / df_tb['cum_evals_fwd_rtr'],
    np.nan,
)

df_tb['TB_ratio_fwd_snr'] = np.where(
    df_tb['cum_evals_fwd_snr'] > 0,
    df_tb['cum_tb_fwd_snr'] / df_tb['cum_evals_fwd_snr'],
    np.nan,
)

# Backward cumulative counts at period end
# If eval_thru_dt is beyond last snapshot, use the last snapshot in-period
period_end_bwd = (
    df_tb.groupby(['pid_pde', 'eval_thru_dt_bwd'])['snpsht_dt']
    .transform('max')
)
is_eval_end_bwd = df_tb['snpsht_dt'] == period_end_bwd

df_tb['cum_evals_bwd_rtr'] = is_eval_end_bwd.groupby(df_tb['pid_pde']).cumsum()
df_tb['cum_tb_bwd_rtr'] = ((df_tb['rtr_rater_box_bwd'] == TB_VALUE) & is_eval_end_bwd).groupby(df_tb['pid_pde']).cumsum()

df_tb['cum_evals_bwd_snr'] = is_eval_end_bwd.groupby(df_tb['pid_pde']).cumsum()
df_tb['cum_tb_bwd_snr'] = ((df_tb['snr_rater_box_bwd'] == TB_VALUE) & is_eval_end_bwd).groupby(df_tb['pid_pde']).cumsum()

# Backward ratios based on eval_thru_dt snapshots
period_totals = (
    df_tb[is_eval_end_bwd]
    .groupby(['pid_pde', 'eval_thru_dt_bwd'])
    .agg(
        total_evals_rtr=('rtr_rater_box_bwd', lambda x: x.notna().sum()),
        total_tb_rtr=('rtr_rater_box_bwd', lambda x: (x == TB_VALUE).sum()),
        total_evals_snr=('snr_rater_box_bwd', lambda x: x.notna().sum()),
        total_tb_snr=('snr_rater_box_bwd', lambda x: (x == TB_VALUE).sum()),
    )
    .reset_index()
)

period_totals['TB_ratio_bwd_rtr'] = np.where(
    period_totals['total_evals_rtr'] > 0,
    period_totals['total_tb_rtr'] / period_totals['total_evals_rtr'],
    np.nan,
)
period_totals['TB_ratio_bwd_snr'] = np.where(
    period_totals['total_evals_snr'] > 0,
    period_totals['total_tb_snr'] / period_totals['total_evals_snr'],
    np.nan,
)

# Paint backward ratios across the rated period

df_tb = df_tb.merge(
    period_totals[['pid_pde', 'eval_thru_dt_bwd', 'TB_ratio_bwd_rtr', 'TB_ratio_bwd_snr']],
    on=['pid_pde', 'eval_thru_dt_bwd'],
    how='left'
)

# -------------------------------------------------------------------
# 4) Pool metrics (rater + senior rater)
# -------------------------------------------------------------------

COMPUTE_POOL_METRICS = True

if COMPUTE_POOL_METRICS:
    # Rater pools (forward)
    rtr_agg = df_tb.groupby(['snpsht_dt', 'rtr_rater_fwd']).agg(
        pool_size_rtr=('pid_pde', 'nunique'),
        pool_mean_fwd_rtr=('TB_ratio_fwd_rtr', 'mean'),
    )
    df_tb = df_tb.merge(rtr_agg, on=['snpsht_dt', 'rtr_rater_fwd'], how='left')

    denom_rtr = (df_tb['pool_size_rtr'] - 1).astype(float)
    valid_rtr = denom_rtr > 0

    df_tb['pool_minus_mean_fwd_rtr'] = np.where(
        valid_rtr,
        (df_tb['pool_size_rtr'] * df_tb['pool_mean_fwd_rtr'] - df_tb['TB_ratio_fwd_rtr']) / denom_rtr,
        np.nan,
    )

    # Rater pools (backward)
    rtr_agg_bwd = df_tb.groupby(['snpsht_dt', 'rtr_rater_bwd']).agg(
        pool_mean_bwd_rtr=('TB_ratio_bwd_rtr', 'mean'),
    )
    df_tb = df_tb.merge(rtr_agg_bwd, on=['snpsht_dt', 'rtr_rater_bwd'], how='left')

    df_tb['pool_minus_mean_bwd_rtr'] = np.where(
        valid_rtr,
        (df_tb['pool_size_rtr'] * df_tb['pool_mean_bwd_rtr'] - df_tb['TB_ratio_bwd_rtr']) / denom_rtr,
        np.nan,
    )

    # Senior rater pools (forward)
    snr_agg = df_tb.groupby(['snpsht_dt', 'snr_rater_fwd']).agg(
        pool_size_snr=('pid_pde', 'nunique'),
        pool_mean_fwd_snr=('TB_ratio_fwd_snr', 'mean'),
    )
    df_tb = df_tb.merge(snr_agg, on=['snpsht_dt', 'snr_rater_fwd'], how='left')

    denom_snr = (df_tb['pool_size_snr'] - 1).astype(float)
    valid_snr = denom_snr > 0

    df_tb['pool_minus_mean_fwd_snr'] = np.where(
        valid_snr,
        (df_tb['pool_size_snr'] * df_tb['pool_mean_fwd_snr'] - df_tb['TB_ratio_fwd_snr']) / denom_snr,
        np.nan,
    )

    # Senior rater pools (backward)
    snr_agg_bwd = df_tb.groupby(['snpsht_dt', 'snr_rater_bwd']).agg(
        pool_mean_bwd_snr=('TB_ratio_bwd_snr', 'mean'),
    )
    df_tb = df_tb.merge(snr_agg_bwd, on=['snpsht_dt', 'snr_rater_bwd'], how='left')

    df_tb['pool_minus_mean_bwd_snr'] = np.where(
        valid_snr,
        (df_tb['pool_size_snr'] * df_tb['pool_mean_bwd_snr'] - df_tb['TB_ratio_bwd_snr']) / denom_snr,
        np.nan,
    )

# Final frame

df_tb


,snpsht_dt,pid_pde,rtr_rater_fwd,snr_rater_fwd,rtr_rater_box_fwd,snr_rater_box_fwd,oer_id_fwd,eval_strt_dt_fwd,eval_thru_dt_fwd,rtr_rater_bwd,...,pool_size_rtr,pool_mean_fwd_rtr,pool_minus_mean_fwd_rtr,pool_mean_bwd_rtr,pool_minus_mean_bwd_rtr,pool_size_snr,pool_mean_fwd_snr,pool_minus_mean_fwd_snr,pool_mean_bwd_snr,pool_minus_mean_bwd_snr
0,2012-10-01,1,RTR_01_08_5,SNR_01_08,30,40,276,2011-09-30,2012-10-01,RTR_01_08_5,...,7,0.000000,0.000000,0.000000,0.000000,35,0.171429,0.176471,0.171429,0.176471
1,2012-10-01,2,RTR_01_27_2,SNR_01_27,70,70,922,2011-09-30,2012-10-01,RTR_01_27_2,...,7,0.142857,0.000000,0.142857,0.000000,35,0.171429,0.147059,0.171429,0.147059
2,2012-10-01,3,RTR_01_01_4,SNR_01_01,60,40,26,2011-09-30,2012-10-01,RTR_01_01_4,...,7,0.000000,0.000000,0.000000,0.000000,35,0.114286,0.117647,0.114286,0.117647
3,2012-10-01,4,RTR_01_15_1,SNR_01_15,70,60,496,2011-09-30,2012-10-01,RTR_01_15_1,...,7,0.428571,0.333333,0.428571,0.333333,35,0.228571,0.235294,0.228571,0.235294
4,2012-10-01,5,RTR_01_23_5,SNR_01_23,30,40,799,2011-09-30,2012-10-01,RTR_01_23_5,...,7,0.000000,0.000000,0.000000,0.000000,35,0.171429,0.176471,0.171429,0.176471
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19995,2017-07-01,996,RTR_05_18_4,SNR_05_18,30,50,4621,2015-09-30,2016-10-01,RTR_06_07_3,...,7,0.200000,0.200000,0.428571,0.500000,35,0.182857,0.176471,0.200000,0.205882
19996,2017-07-01,997,RTR_05_11_5,SNR_05_11,60,70,4380,2015-09-30,2016-10-01,RTR_06_25_3,...,7,0.171429,0.166667,0.571429,0.666667,35,0.234286,0.229412,0.285714,0.294118
19997,2017-07-01,998,RTR_05_06_1,SNR_05_06,60,60,4176,2015-09-30,2016-10-01,RTR_06_18_1,...,7,0.228571,0.166667,0.142857,0.166667,35,0.200000,0.200000,0.314286,0.323529
19998,2017-07-01,999,RTR_05_07_3,SNR_05_07,40,40,4227,2015-09-30,2016-10-01,RTR_06_03_5,...,7,0.171429,0.200000,0.428571,0.500000,35,0.154286,0.158824,0.228571,0.235294


In [198]:
pid = 551
cols = df_tb.columns.tolist()
cols = [col for col in cols if 'rtr' not in col]

df_oers2 = df_oers.drop('rtr_rater_box', axis=1)
df_oers2[df_oers2.pid_pde == pid].sort_values(by='eval_thru_dt')


,pid_pde,rater,snr_rater,snr_rater_box,oer_id,eval_strt_dt,eval_thru_dt
234,551,RTR_01_07_4,SNR_01_07,50,235,2011-09-30,2012-10-01
1624,551,RTR_02_18_5,SNR_02_18,70,1629,2012-09-30,2013-10-01
2431,551,RTR_03_13_2,SNR_03_13,60,2432,2013-09-30,2014-10-01
3998,551,RTR_04_29_5,SNR_04_29,50,3999,2014-09-30,2015-10-01
4837,551,RTR_05_24_5,SNR_05_24,70,4837,2015-09-30,2016-10-01
5898,551,RTR_06_26_4,SNR_06_26,30,5903,2016-09-30,2017-10-01


In [199]:
cols2 = ['pid_pde', 'snpsht_dt',
    'oer_id_bwd', 
    'snr_rater_bwd', 'snr_rater_box_bwd',
    'cum_tb_bwd_snr', 'cum_evals_bwd_snr', 'pool_size_snr','TB_ratio_bwd_snr', 'pool_mean_bwd_snr', 'pool_minus_mean_bwd_snr', 
    'oer_id_fwd', 
    'snr_rater_fwd', 'snr_rater_box_fwd',
    'cum_tb_fwd_snr', 'cum_evals_fwd_snr', 'TB_ratio_fwd_snr', 'pool_mean_fwd_snr', 'pool_minus_mean_fwd_snr', ]    

df_tb[df_tb.pid_pde == pid][cols2].sort_values(by='snpsht_dt')


,pid_pde,snpsht_dt,oer_id_bwd,snr_rater_bwd,snr_rater_box_bwd,cum_tb_bwd_snr,cum_evals_bwd_snr,pool_size_snr,TB_ratio_bwd_snr,pool_mean_bwd_snr,pool_minus_mean_bwd_snr,oer_id_fwd,snr_rater_fwd,snr_rater_box_fwd,cum_tb_fwd_snr,cum_evals_fwd_snr,TB_ratio_fwd_snr,pool_mean_fwd_snr,pool_minus_mean_fwd_snr
550,551,2012-10-01,235.0,SNR_01_07,50.0,0,1,35,0.0,0.257143,0.264706,235,SNR_01_07,50,0,1,0.000000,0.257143,0.264706
1550,551,2013-01-01,1629.0,SNR_02_18,70.0,0,1,35,1.0,0.171429,0.147059,235,SNR_01_07,50,0,1,0.000000,0.257143,0.264706
2550,551,2013-04-01,1629.0,SNR_02_18,70.0,0,1,35,1.0,0.171429,0.147059,235,SNR_01_07,50,0,1,0.000000,0.257143,0.264706
3550,551,2013-07-01,1629.0,SNR_02_18,70.0,0,1,35,1.0,0.171429,0.147059,235,SNR_01_07,50,0,1,0.000000,0.257143,0.264706
4550,551,2013-10-01,1629.0,SNR_02_18,70.0,1,2,35,1.0,0.171429,0.147059,1629,SNR_02_18,70,1,2,0.500000,0.157143,0.147059
5550,551,2014-01-01,2432.0,SNR_03_13,60.0,1,2,35,0.0,0.257143,0.264706,1629,SNR_02_18,70,1,2,0.500000,0.157143,0.147059
6550,551,2014-04-01,2432.0,SNR_03_13,60.0,1,2,35,0.0,0.257143,0.264706,1629,SNR_02_18,70,1,2,0.500000,0.157143,0.147059
7550,551,2014-07-01,2432.0,SNR_03_13,60.0,1,2,35,0.0,0.257143,0.264706,1629,SNR_02_18,70,1,2,0.500000,0.157143,0.147059
8550,551,2014-10-01,2432.0,SNR_03_13,60.0,1,3,35,0.0,0.257143,0.264706,2432,SNR_03_13,60,1,3,0.333333,0.238095,0.235294
9550,551,2015-01-01,3999.0,SNR_04_29,50.0,1,3,35,0.0,0.100000,0.102941,2432,SNR_03_13,60,1,3,0.333333,0.238095,0.235294
